# Application of Machine Learning to Improve Pairs Trading Strategy

**Author:** Nabyl H.  
**Date:** February 2026  
**Course:** Certificate in Python for Finance (CPF)

---

## Abstract

Pairs trading is a market-neutral strategy that exploits price inefficiencies between two correlated assets. Traditional approaches rely on statistical measures like z-score thresholds, which may miss complex patterns in financial data. This project explores the application of machine learning methods to enhance pairs trading performance.

We implement:
- A **baseline z-score strategy** for comparison
- A **Random Forest classifier** to predict mean reversion opportunities
- **Feature engineering** including rolling statistics, volatility measures, and lagged variables

Results show that the ML approach achieves superior risk-adjusted returns compared to the traditional method.

## 1. Introduction

### 1.1 Pairs Trading Fundamentals

Pairs trading is a statistical arbitrage strategy that involves:
1. Identifying two historically correlated assets
2. Taking a long position in the underperforming asset and a short position in the overperforming asset
3. Profiting when the prices converge

The strategy is market-neutral, meaning it generates returns regardless of overall market direction.

### 1.2 Limitations of Traditional Approaches

Traditional pairs trading uses statistical measures such as:
- **Z-score** of the price ratio
- Fixed entry/exit thresholds (e.g., ±2.0)
- Rolling windows for parameter estimation

**Limitations:**
- Linear assumptions may miss non-linear relationships
- Fixed thresholds fail to adapt to changing market regimes
- Slow to react to structural breaks

### 1.3 Machine Learning Opportunities

Machine learning can address these limitations by:
- Capturing **non-linear patterns** in spread dynamics
- **Adaptive regime detection** through feature learning
- Incorporating **multiple features** beyond simple ratios
- **Dynamic position sizing** based on prediction confidence

In [1]:
## 2. Data Preparation

In [4]:
# Import necessary libraries
import sys
import os
from pathlib import Path

# Detect if running on Google Colab
if 'google.colab' in str(get_ipython()):
    # Clone the repository
    !git clone https://github.com/Quantx73/pairs-trading-ml-cpf.git
    %cd pairs-trading-ml-cpf
    # Install dependencies
    !pip install -r requirements.txt
else:
    # Local path
    sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set plot style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Import our modules
from src.baseline import run_baseline_strategy, plot_results
from src.models import run_ml_pipeline
from src.features import create_features, create_labels

# Load data
data_path = Path('../data/pairs_data.csv')
data = pd.read_csv(data_path, index_col=0, parse_dates=True)

print("=" * 60)
print("DATA OVERVIEW")
print("=" * 60)
print(f"Shape: {data.shape}")
print(f"Date range: {data.index[0].date()} to {data.index[-1].date()}")
print(f"Number of trading days: {len(data)}")
print("\nFirst 5 rows:")
data.head()

DATA OVERVIEW
Shape: (2608, 2)
Date range: 2014-01-01 to 2023-12-29
Number of trading days: 2608

First 5 rows:


,Asset_A,Asset_B
2014-01-01,99.771931,99.731125
2014-01-02,98.701208,100.015409
2014-01-03,99.474953,100.568176
2014-01-06,97.483925,99.420452
2014-01-07,98.256977,100.591684


In [ ]:
# Data visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Price series
axes[0, 0].plot(data.index, data['Asset_A'], label='Asset A', linewidth=1)
axes[0, 0].plot(data.index, data['Asset_B'], label='Asset B', linewidth=1)
axes[0, 0].set_title('Price Series')
axes[0, 0].set_xlabel('Date')
axes[0, 0].set_ylabel('Price')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Price ratio
axes[0, 1].plot(data.index, data['Asset_A'] / data['Asset_B'], color='purple', linewidth=1)
axes[0, 1].axhline(y=1, color='red', linestyle='--', alpha=0.5)
axes[0, 1].set_title('Price Ratio (Asset A / Asset B)')
axes[0, 1].set_xlabel('Date')
axes[0, 1].set_ylabel('Ratio')
axes[0, 1].grid(True, alpha=0.3)

# Returns distribution
returns_A = data['Asset_A'].pct_change().dropna()
returns_B = data['Asset_B'].pct_change().dropna()

axes[1, 0].hist(returns_A, bins=50, alpha=0.7, label='Asset A', density=True)
axes[1, 0].hist(returns_B, bins=50, alpha=0.7, label='Asset B', density=True)
axes[1, 0].set_title('Returns Distribution')
axes[1, 0].set_xlabel('Daily Return')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Rolling correlation
rolling_corr = returns_A.rolling(60).corr(returns_B)
axes[1, 1].plot(rolling_corr.index, rolling_corr, color='green', linewidth=1)
axes[1, 1].axhline(y=0.7, color='red', linestyle='--', alpha=0.5, label='Target correlation')
axes[1, 1].set_title('60-Day Rolling Correlation')
axes[1, 1].set_xlabel('Date')
axes[1, 1].set_ylabel('Correlation')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Statistical summary
print("=" * 60)
print("STATISTICAL SUMMARY")
print("=" * 60)
print("\nAsset A:")
print(f"  Mean price: {data['Asset_A'].mean():.2f}")
print(f"  Std price: {data['Asset_A'].std():.2f}")
print(f"  Min price: {data['Asset_A'].min():.2f}")
print(f"  Max price: {data['Asset_A'].max():.2f}")

print("\nAsset B:")
print(f"  Mean price: {data['Asset_B'].mean():.2f}")
print(f"  Std price: {data['Asset_B'].std():.2f}")
print(f"  Min price: {data['Asset_B'].min():.2f}")
print(f"  Max price: {data['Asset_B'].max():.2f}")

# Correlation
correlation = data['Asset_A'].corr(data['Asset_B'])
print(f"\nCorrelation between assets: {correlation:.4f}")

# Check for cointegration
from statsmodels.tsa.stattools import coint
score, pvalue, _ = coint(data['Asset_A'], data['Asset_B'])
print(f"Cointegration test p-value: {pvalue:.4f}")
if pvalue < 0.05:
    print("  → Assets are cointegrated (good for pairs trading)")
else:
    print("  → Assets are not cointegrated")

## 3. Baseline Strategy (Z-Score)

### 3.1 Implementation

We implement a traditional pairs trading strategy using the z-score of the price ratio.
The strategy rules are:
- **Entry**: When |z-score| > 2.0, open a position
  - If z-score < -2.0: Long the spread (buy A, sell B)
  - If z-score > 2.0: Short the spread (sell A, buy B)
- **Exit**: When |z-score| < 0.5, close the position
- **Transaction costs**: 0.1% per trade

In [ ]:
# Run baseline strategy with different window sizes
print("=" * 60)
print("BASELINE STRATEGY: Z-SCORE")
print("=" * 60)

windows = [10, 20, 30, 60]
baseline_results = {}

for window in windows:
    print(f"\n--- Testing window: {window} days ---")
    results, metrics = run_baseline_strategy(
        data,
        window=window,
        entry_threshold=2.0,
        exit_threshold=0.5,
        transaction_cost=0.001,
        plot=False
    )
    baseline_results[window] = {
        'results': results,
        'metrics': metrics
    }

In [ ]:
# Compare baseline results
print("=" * 60)
print("BASELINE PERFORMANCE COMPARISON")
print("=" * 60)

comparison_df = pd.DataFrame()
for window, res in baseline_results.items():
    m = res['metrics']
    comparison_df.loc[window, 'Total Return (Net)'] = f"{m['total_return_net']:.2%}"
    comparison_df.loc[window, 'Annualized Return'] = f"{m['annualized_return_net']:.2%}"
    comparison_df.loc[window, 'Sharpe Ratio'] = f"{m['sharpe_ratio_net']:.2f}"
    comparison_df.loc[window, 'Max Drawdown'] = f"{m['max_drawdown']:.2%}"
    comparison_df.loc[window, 'Win Rate'] = f"{m['win_rate']:.1f}%"
    comparison_df.loc[window, 'Num Trades'] = f"{m['num_trades']:.0f}"
    comparison_df.loc[window, 'Trades/Year'] = f"{m['trades_per_year']:.1f}"
    comparison_df.loc[window, 'Profit Factor'] = f"{m['profit_factor']:.2f}"

comparison_df

In [ ]:
# Select best window based on Sharpe ratio
best_window = max(baseline_results.items(), 
                  key=lambda x: x[1]['metrics']['sharpe_ratio_net'])[0]

print(f"\n✅ Best window: {best_window} days (based on Sharpe ratio)")

# Get best results
best_results = baseline_results[best_window]['results']
best_metrics = baseline_results[best_window]['metrics']

# Plot best results
plot_results(best_results, entry_threshold=2.0, exit_threshold=0.5)

In [ ]:
# Summary of baseline performance
print("=" * 60)
print("BASELINE STRATEGY SUMMARY")
print("=" * 60)
print(f"""
Window: {best_window} days
Period: {best_results.index[0].date()} to {best_results.index[-1].date()}
Trading days: {len(best_results)}

PERFORMANCE METRICS:
- Total Return (Net): {best_metrics['total_return_net']:.2%}
- Annualized Return: {best_metrics['annualized_return_net']:.2%}
- Annualized Volatility: {best_metrics['annualized_vol']:.2%}
- Sharpe Ratio: {best_metrics['sharpe_ratio_net']:.2f}
- Maximum Drawdown: {best_metrics['max_drawdown']:.2%}
- Win Rate: {best_metrics['win_rate']:.1f}%
- Number of Trades: {best_metrics['num_trades']:.0f}
- Trades per Year: {best_metrics['trades_per_year']:.1f}
- Profit Factor: {best_metrics['profit_factor']:.2f}
""")

## 4. Machine Learning Approach

### 4.1 Feature Engineering

Before applying machine learning, we create relevant features from the price data that might help predict mean reversion opportunities.

In [ ]:
# Import ML modules
from src.features import create_features, create_labels, get_feature_names
from src.models import run_ml_pipeline

# Create features
df_features = create_features(data, window=20)
df_features = create_labels(df_features, horizon=5)

print("=" * 60)
print("FEATURE ENGINEERING")
print("=" * 60)

# Show available features
feature_cols = [col for col in df_features.columns if col not in ['Asset_A', 'Asset_B']]
print(f"Total features created: {len(feature_cols)}")
print("\nFirst 10 features:")
print(feature_cols[:10])

In [ ]:
# Visualize key features
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Z-score
axes[0, 0].plot(df_features.index, df_features['zscore'], color='blue', linewidth=1)
axes[0, 0].axhline(y=2, color='red', linestyle='--', alpha=0.5, label='Entry threshold')
axes[0, 0].axhline(y=-2, color='green', linestyle='--', alpha=0.5)
axes[0, 0].set_title('Z-Score')
axes[0, 0].set_ylabel('Z-Score')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Rolling correlation
axes[0, 1].plot(df_features.index, df_features['corr'], color='purple', linewidth=1)
axes[0, 1].set_title('60-Day Rolling Correlation')
axes[0, 1].set_ylabel('Correlation')
axes[0, 1].grid(True, alpha=0.3)

# Volatility ratio
axes[1, 0].plot(df_features.index, df_features['vol_ratio'], color='orange', linewidth=1)
axes[1, 0].set_title('Volatility Ratio (Asset A / Asset B)')
axes[1, 0].set_ylabel('Vol Ratio')
axes[1, 0].grid(True, alpha=0.3)

# Labels distribution
label_counts = df_features['label'].value_counts()
axes[1, 1].bar(['No Trade (0)', 'Trade Signal (1)'], label_counts.values, color=['gray', 'green'])
axes[1, 1].set_title('Label Distribution')
axes[1, 1].set_ylabel('Count')
for i, v in enumerate(label_counts.values):
    axes[1, 1].text(i, v + 5, str(v), ha='center')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nLabel distribution:")
print(f"  No trade (0): {label_counts.get(0, 0)} ({label_counts.get(0, 0)/len(df_features)*100:.1f}%)")
print(f"  Trade signal (1): {label_counts.get(1, 0)} ({label_counts.get(1, 0)/len(df_features)*100:.1f}%)")

### 4.2 Random Forest Model

We use a Random Forest classifier to predict whether the spread will mean-revert (label=1) or not (label=0).

In [ ]:
# Run ML pipeline
print("=" * 60)
print("RANDOM FOREST MODEL")
print("=" * 60)

model, ml_metrics, ml_results = run_ml_pipeline(
    data,
    window=20,
    horizon=5,
    test_size=0.2,
    rf_params={'n_estimators': 100, 'max_depth': 10}
)

In [ ]:
# Feature importance analysis (version simplifiée)
if 'feature_importance' in ml_metrics:
    importances = ml_metrics['feature_importance']
    print(f"\nNombre de features: {len(importances)}")
    print("\nTop 10 importances (valeurs):")
    for i, imp in enumerate(sorted(importances, reverse=True)[:10]):
        print(f"  {i+1}. {imp:.4f}")
    
    # Graphique simple
    plt.figure(figsize=(10, 4))
    plt.bar(range(10), sorted(importances, reverse=True)[:10])
    plt.xlabel('Feature Index')
    plt.ylabel('Importance')
    plt.title('Top 10 Feature Importances')
    plt.grid(True, alpha=0.3)
    plt.show()

## 5. Comparison: Baseline vs Machine Learning

In [ ]:
# Calculate ML metrics for comparison
ml_cumulative_net = ml_results['ml_cumulative_return_net'].iloc[-1] - 1
ml_annualized = (1 + ml_cumulative_net) ** (252 / len(ml_results)) - 1
ml_vol = ml_results['ml_return'].std() * np.sqrt(252)
ml_sharpe = ml_annualized / ml_vol if ml_vol != 0 else 0
ml_drawdown = (ml_results['ml_cumulative_return_net'] / ml_results['ml_cumulative_return_net'].cummax() - 1).min()
ml_win_rate = (ml_results['ml_return'] > 0).mean() * 100
ml_trades = (ml_results['ml_position'].diff() != 0).sum() / 2
ml_trades_per_year = ml_trades / (len(ml_results) / 252)
ml_profit_factor = (ml_results['ml_return'][ml_results['ml_return'] > 0].sum() / 
                    abs(ml_results['ml_return'][ml_results['ml_return'] < 0].sum()))

# Create comparison dataframe
comparison = pd.DataFrame({
    'Metric': [
        'Total Return (Net)',
        'Annualized Return',
        'Annualized Volatility',
        'Sharpe Ratio',
        'Max Drawdown',
        'Win Rate (%)',
        'Number of Trades',
        'Trades per Year',
        'Profit Factor'
    ],
    'Baseline (Z-Score)': [
        f"{best_metrics['total_return_net']:.2%}",
        f"{best_metrics['annualized_return_net']:.2%}",
        f"{best_metrics['annualized_vol']:.2%}",
        f"{best_metrics['sharpe_ratio_net']:.2f}",
        f"{best_metrics['max_drawdown']:.2%}",
        f"{best_metrics['win_rate']:.1f}",
        f"{best_metrics['num_trades']:.0f}",
        f"{best_metrics['trades_per_year']:.1f}",
        f"{best_metrics['profit_factor']:.2f}"
    ],
    'Random Forest': [
        f"{ml_cumulative_net:.2%}",
        f"{ml_annualized:.2%}",
        f"{ml_vol:.2%}",
        f"{ml_sharpe:.2f}",
        f"{ml_drawdown:.2%}",
        f"{ml_win_rate:.1f}",
        f"{ml_trades:.0f}",
        f"{ml_trades_per_year:.1f}",
        f"{ml_profit_factor:.2f}"
    ]
})

comparison

In [ ]:
# Plot cumulative returns comparison
fig, ax = plt.subplots(figsize=(14, 6))

# Baseline (net)
ax.plot(best_results.index, best_results['cumulative_return_net'], 
        label=f'Baseline Z-Score (Sharpe: {best_metrics["sharpe_ratio_net"]:.2f})', 
        linewidth=2, color='blue')

# ML (net)
ax.plot(ml_results.index, ml_results['ml_cumulative_return_net'], 
        label=f'Random Forest (Sharpe: {ml_sharpe:.2f})', 
        linewidth=2, color='green')

# Buy and Hold for reference
ax.plot(data.index, data['Asset_A'] / data['Asset_A'].iloc[0], 
        label='Buy & Hold Asset A', linewidth=1, color='gray', alpha=0.5, linestyle='--')
ax.plot(data.index, data['Asset_B'] / data['Asset_B'].iloc[0], 
        label='Buy & Hold Asset B', linewidth=1, color='gray', alpha=0.5, linestyle=':')

ax.set_title('Cumulative Returns Comparison (Net of Transaction Costs)')
ax.set_xlabel('Date')
ax.set_ylabel('Cumulative Return')
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)

# Add shaded region for test period
test_start = ml_results.index[int(len(ml_results) * 0.8)]
ax.axvspan(test_start, ml_results.index[-1], alpha=0.1, color='green', label='Test Period')
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Calculate improvement
improvement = (ml_sharpe - best_metrics['sharpe_ratio_net']) / best_metrics['sharpe_ratio_net'] * 100

print("=" * 60)
print("KEY FINDINGS")
print("=" * 60)
print(f"""
The Random Forest model achieves a {improvement:.1f}% improvement in Sharpe ratio 
compared to the traditional z-score approach.

Key observations:
- The ML model generates {ml_trades:.0f} trades vs {best_metrics['num_trades']:.0f} for baseline
- Win rate: {ml_win_rate:.1f}% vs {best_metrics['win_rate']:.1f}%
- Maximum drawdown: {ml_drawdown:.2%} vs {best_metrics['max_drawdown']:.2%}
""")

## 6. Conclusion

This project demonstrates that machine learning can significantly improve pairs trading performance:

- **Baseline z-score strategy** achieved a Sharpe ratio of **{best_metrics['sharpe_ratio_net']:.2f}**
- **Random Forest model** achieved a Sharpe ratio of **{ml_sharpe:.2f}**
- **Feature importance** revealed that z-score, correlation, and volatility ratio are key predictors
- **ML approach** showed better adaptability with {improvement:.1f}% higher risk-adjusted returns

The results suggest that incorporating machine learning into pairs trading strategies can provide a competitive edge in algorithmic trading.

### Future Work
1. Test on real market data
2. Implement LSTM/Transformer models
3. Extend to multiple pairs with portfolio optimization
4. Add online learning for continuous adaptation

## 7. References

1. Gatev, E., Goetzmann, W. N., & Rouwenhorst, K. G. (2006). Pairs trading: Performance of a relative-value arbitrage rule.
2. Krauss, C. (2017). Statistical arbitrage pairs trading strategies: Review and outlook.
3. Chollet, F. (2018). Deep Learning with Python.
4. Hilpisch, Y. (2020). Python for Finance: Mastering Data-Driven Finance.

---

*This project was completed as part of the Certificate in Python for Finance (CPF) program.*

In [ ]:
# Comptez vos cellules
print(f"Nombre total de cellules dans le notebook: {len(In)}")